# Assignment 15: Instruction Tuning

Fill in every `# TODO` and answer every question in the **Your answer** cells. Run all cells before submitting.

Before you start, put these files in the same folder as this notebook: `tiny_gpt.py`, `base_model.pt` (yours from Assignment 14, or the provided one) and `tinystories.txt` (from Assignment 14).

## Setup

`tiny_gpt.py` (in the same folder as this notebook) contains the model you built in Assignment 14, plus `train`, `generate`, `save_model` and `load_model`.

Put your `base_model.pt` from Assignment 14 in this folder. If your model did not train well, use the `base_model.pt` provided with this assignment.

In [ ]:
import os
import re
import random
import urllib.request
import torch
import matplotlib.pyplot as plt
from tiny_gpt import TinyGPT, encode, decode, clean, train, generate, estimate_loss, save_model, load_model

torch.manual_seed(0)
random.seed(0)
base = load_model("base_model.pt")
print(f"Base model: {sum(p.numel() for p in base.parameters()):,} parameters, context {base.context_size} characters")

## Step 1. The instruction data

A base model only continues text. To make it **follow an instruction**, we fine-tune it on examples of the form *instruction → response*. Real chat models are fine-tuned on hundreds of thousands of such examples written by people. Ours is much simpler.

**The task:** given a topic, write the beginning of a story about that topic. Each training example looks like this:

```
Topic: dog
Story: Once upon a time, there was a little dog named Max. He loved to run and play in the park.
```

We build the examples from new TinyStories text (stories the base model never saw): take the first sentence(s) of each story, up to about 110 characters, and keep it if it mentions **exactly one** of our 10 topics. The cell downloads the first 40 MB of the TinyStories training file (about 1 minute).

In [ ]:
TOPICS = ["dog", "cat", "bird", "bear", "fish", "frog", "ball", "cake", "car", "tree"]

if not os.path.exists("stories_40mb.txt"):
    url = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt"
    request = urllib.request.Request(url, headers={"Range": "bytes=0-40000000"})   # only the first 40 MB
    with urllib.request.urlopen(request) as response, open("stories_40mb.txt", "wb") as f:
        f.write(response.read())

with open("stories_40mb.txt", encoding="utf-8", errors="ignore") as f:
    stories = f.read().split("<|endoftext|>")
print("Number of stories:", len(stories))

### Question 1. Build the examples

a. Complete `mentions(topic, text)`: return `True` if the word `topic` or its plural (`topic + "s"`) appears in `text` as a **whole word**, ignoring case. *Hint:* `re.search(rf"\b{topic}s?\b", text, re.IGNORECASE)`.

b. Complete `make_example(topic, story)`: return the training text for one example, in the format shown above, followed by a blank line (`"\n\n"`) that marks the end of the response.

c. How many examples are there for each topic? We keep at most 400 per topic. Why is it a good idea to balance the topics?

In [ ]:
def mentions(topic, text):
    # TODO: whole-word match of topic or its plural, ignoring case

def prompt_for(topic):
    return f"Topic: {topic}\nStory: "

def make_example(topic, story):
    # TODO: the prompt, the story, and a blank line

by_topic = {t: [] for t in TOPICS}
for story in stories:
    first = clean(story.strip().split("\n")[0])           # first paragraph, vocabulary characters only
    if len(first) < 60:
        continue
    first = first[:110].rsplit(" ", 1)[0]                   # about 110 characters, cut at a space
    found = [t for t in TOPICS if mentions(t, first)]
    if len(found) == 1:
        by_topic[found[0]].append(first)

print({t: len(v) for t, v in by_topic.items()})
examples = [(t, s) for t, v in by_topic.items() for s in v[:400]]
random.shuffle(examples)
print("Examples used:", len(examples))
print()
print(make_example(*examples[0]))

**Your answer:** *Write your answer here.*

## Step 2. How does the base model do?

`on_topic_rate` gives the model each topic's prompt 10 times and measures how often the story it writes mentions the topic.

In [ ]:
def on_topic_rate(model, repeats=10, seed=1):
    torch.manual_seed(seed)
    rates = {}
    for t in TOPICS:
        hits = sum(mentions(t, generate(model, prompt_for(t), 120)) for _ in range(repeats))
        rates[t] = hits / repeats
    return rates

base_rates = on_topic_rate(base)
print(f"Base model on-topic rate: {sum(base_rates.values()) / len(TOPICS):.0%}")
print(base_rates)
for t in ["dog", "cake"]:
    print(f"\n{prompt_for(t)}{generate(base, prompt_for(t), 120)}")

### Question 2. The base model

a. What is the base model's on-topic rate? Why is it so low, even though it can write decent stories?

b. Look at what the base model writes after `Story: `. Does it at least produce a story?

**Your answer:** *Write your answer here.*

## Step 3. Instruction tuning

Fine-tuning is just **more training**, on new data, starting from the pre-trained weights. Two things differ from pre-training:

- Each training sequence is **one whole example**, starting at `Topic:`, so the model always sees the instruction.
- The loss is computed **only on the response**. We don't want to teach the model to write instructions (the user does that); we want it to learn what to write *after* an instruction.

### Question 3. Mask the prompt

For one example, the input `x` is the token IDs of prompt + response without the last one, and the target `y` is the same IDs shifted by one, as in pre-training. To leave the prompt out of the loss, set the targets that are still **inside the prompt** to `-100`: PyTorch's `cross_entropy` ignores targets equal to `-100`.

Complete `make_xy`. If the prompt has `p` tokens, the first `p - 1` targets are prompt characters (`y[p-1]` is the first character of the response).

Then answer: in the example printed below, how many targets are masked, and which character is the first one that counts?

In [ ]:
def make_xy(topic, story, context_size=base.context_size):
    prompt_ids = encode(prompt_for(topic))
    ids = (prompt_ids + encode(story + "\n\n"))[:context_size + 1]   # must fit in the context
    x, y = ids[:-1], ids[1:]
    # TODO: replace the first len(prompt_ids) - 1 targets by -100
    return x, y

x, y = make_xy(*examples[0])
print("input :", repr(decode(x)))
print("target:", [decode([t]) if t != -100 else "-" for t in y][:25], "...")
print("masked targets:", y.count(-100))

**Your answer:** *Write your answer here.*

### Question 4. Fine-tune

`sft_batch` picks random examples and pads them to the same length (padding targets are also `-100`). Complete the training loop, then train a copy of the base model for 1,500 steps with learning rate 0.001 (about 4–6 minutes).

In [ ]:
import copy

pairs = [make_xy(t, s) for t, s in examples]

def sft_batch(batch_size=32):
    chosen = random.sample(pairs, batch_size)
    length = max(len(x) for x, _ in chosen)
    X = torch.zeros(batch_size, length, dtype=torch.long)
    Y = torch.full((batch_size, length), -100)
    for i, (x, y) in enumerate(chosen):
        X[i, :len(x)] = torch.tensor(x)
        Y[i, :len(y)] = torch.tensor(y)
    return X, Y

instruct = copy.deepcopy(base)
optimizer = torch.optim.AdamW(instruct.parameters(), lr=1e-3)
losses = []
for step in range(1500):
    # TODO: one training step on an sft_batch
    losses.append(loss.item())
    if step % 250 == 0:
        print(f"step {step:5d}: loss on responses {loss.item():.3f}")

plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss on the responses"); plt.show()
save_model(instruct, "instruct_model.pt")

### Question 5. Did it learn to follow the instruction?

Compute the instruction-tuned model's on-topic rate and plot the two models' rates per topic side by side. Show one story for each of 3 topics.

In [ ]:
# TODO: compute instruct_rates with on_topic_rate
print(f"Base model:              {sum(base_rates.values()) / len(TOPICS):.0%} on topic")
print(f"Instruction-tuned model: {sum(instruct_rates.values()) / len(TOPICS):.0%} on topic")

x = range(len(TOPICS))
plt.bar([i - 0.2 for i in x], base_rates.values(), width=0.4, label="base")
plt.bar([i + 0.2 for i in x], instruct_rates.values(), width=0.4, label="instruction-tuned")
plt.xticks(x, TOPICS); plt.ylabel("on-topic rate"); plt.legend(); plt.show()

torch.manual_seed(3)
for t in ["dog", "frog", "cake"]:
    print(f"{prompt_for(t)}{generate(instruct, prompt_for(t), 120)}\n")

**Your answer:** *Write your answer here.*

### Question 6. Did it forget how to write stories?

Fine-tuning on a narrow task can make a model worse at what it knew before (**catastrophic forgetting**). Compare the loss of the base and the instruction-tuned models on plain TinyStories text (the last 10% of the Assignment 14 data). Did the loss go up? By how much?

In [ ]:
with open("tinystories.txt", encoding="utf-8") as f:     # the file from Assignment 14 (copy it into this folder)
    plain = f.read()
plain_val = torch.tensor(encode(plain[int(0.9 * len(plain)):]))

torch.manual_seed(0)
# TODO: estimate the loss of both models on plain_val
print(f"Loss on plain stories: base {base_loss:.3f}, instruction-tuned {instruct_loss:.3f}")

**Your answer:** *Write your answer here.*

### Question 7. A topic it has never seen

Try topics that were **not** in the training data, such as `horse`, `moon` and `boat`. Does the instruction-tuned model follow them? Explain why or why not. How is this different from a real chat model?

In [ ]:
torch.manual_seed(4)
for t in ["horse", "moon", "boat"]:
    outs = [generate(instruct, prompt_for(t), 120) for _ in range(10)]
    print(f"{t}: on topic in {sum(mentions(t, o) for o in outs)}/10.  Example: {outs[0]}")

**Your answer:** *Write your answer here.*

### Question 8. Reflection

a. When the instructor tried this assignment **without** the prompt masking of Question 3 (training on random 128-character windows of the instruction text, as in pre-training), the on-topic rate only reached about 17%. Give two reasons why masking the prompt and training on whole examples works better.

b. In 3–4 sentences, describe how instruction tuning for a real chat model (like ChatGPT or Claude) is similar to what you did, and how it is different.

**Your answer:** *Write your answer here.*